In [ ]:
!pip install rasterio

In [ ]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import numpy as np
import pandas as pd
import os
from shapely.geometry import mapping

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
def sample_from_shapefile(raster_path, shapefile_path, label):
    """Extract pixel values under shapefile polygons and assign label."""
    with rasterio.open(raster_path) as src:
        gdf = gpd.read_file(shapefile_path)
        samples = []
        for _, row in gdf.iterrows():
            geom = [mapping(row['geometry'])]
            try:
                out_image, _ = mask(src, geom, crop=True)
                arr = out_image.reshape(src.count, -1).T
                arr = arr[~np.isnan(arr).any(axis=1)]  # remove NaN pixels
                lbls = np.full(arr.shape[0], label, dtype=np.uint8)
                samples.append((arr, lbls))
            except Exception as e:
                print(f"⚠️ Skipping polygon due to error: {e}")
        if samples:
            X = np.vstack([s[0] for s in samples])
            y = np.hstack([s[1] for s in samples])
            return X, y
        else:
            return np.empty((0, src.count)), np.empty((0,))

In [ ]:
aois = ["barlacha", "khanchengyao", "langpo", "samundraTapu",
        "vasuki", "fanchan", "gepang", "karzok"]

In [ ]:
X_all, y_all = [], []

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
for aoi in aois:
    print(f"\n🔹 Processing AOI: {aoi}")

    x_path = f"/content/drive/MyDrive/GEE_Exports/Predicted/{aoi}_x_train_enhanced.tif"
    lake_shp = f"/content/drive/MyDrive/GEE_Exports/Shapefiles/{aoi}/{aoi}_lake.shp"
    nonlake_shp = f"/content/drive/MyDrive/GEE_Exports/Shapefiles/{aoi}/{aoi}_non_lake.shp"

    # Extract samples
    X_lake, y_lake = sample_from_shapefile(x_path, lake_shp, 1)
    X_nonlake, y_nonlake = sample_from_shapefile(x_path, nonlake_shp, 0)

    print(f"  Lake samples: {len(y_lake)}, Non-lake samples: {len(y_nonlake)}")

    # Balance classes (downsample majority)
    min_len = min(len(y_lake), len(y_nonlake))
    X_balanced = np.vstack([X_lake[:min_len], X_nonlake[:min_len]])
    y_balanced = np.hstack([y_lake[:min_len], y_nonlake[:min_len]])

    # Collect
    X_all.append(X_balanced)
    y_all.append(y_balanced)


🔹 Processing AOI: barlacha
  Lake samples: 7391, Non-lake samples: 28471

🔹 Processing AOI: khanchengyao
  Lake samples: 157402, Non-lake samples: 965297

🔹 Processing AOI: langpo
  Lake samples: 180458, Non-lake samples: 1540074

🔹 Processing AOI: samundraTapu
  Lake samples: 30576, Non-lake samples: 47682

🔹 Processing AOI: vasuki
  Lake samples: 2352, Non-lake samples: 3869

🔹 Processing AOI: fanchan
  Lake samples: 1064, Non-lake samples: 1658

🔹 Processing AOI: gepang
  Lake samples: 28994, Non-lake samples: 39569

🔹 Processing AOI: karzok
  Lake samples: 3546043, Non-lake samples: 12868233


In [ ]:
X_all = np.vstack(X_all)
y_all = np.hstack(y_all)

In [ ]:
print("\nFinal training dataset:", X_all.shape, y_all.shape)


Final training dataset: (7908560, 11) (7908560,)


In [ ]:
feature_names = [f"band_{i+1}" for i in range(X_all.shape[1])]
df = pd.DataFrame(X_all, columns=feature_names)
df["label"] = y_all

In [ ]:
csv_path = "/content/drive/MyDrive/GEE_Exports/glacial_lake_dataset_polygon.csv"
df.to_csv(csv_path, index=False)
print("Flattened dataset saved as CSV:", csv_path)

Flattened dataset saved as CSV: /content/drive/MyDrive/GEE_Exports/glacial_lake_dataset_polygon.csv


# Training

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=23,
    max_features="sqrt",
    min_samples_leaf=7,
    class_weight="balanced",
    bootstrap=True,
    criterion="gini",
    oob_score=True,      # OOB validation
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=23,
                       min_samples_leaf=7, n_estimators=250, n_jobs=-1,
                       oob_score=True, random_state=42)

In [ ]:
print("\n📊 Random Forest trained!")
print("OOB Score:", rf.oob_score_)
print(classification_report(y_val, rf.predict(X_val)))
print(confusion_matrix(y_val, rf.predict(X_val)))

In [ ]:
import matplotlib.pyplot as plt

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8,5))
plt.bar(range(X_all.shape[1]), importances[indices], align="center")
plt.xticks(range(X_all.shape[1]), [feature_names[i] for i in indices], rotation=45)
plt.title("Feature Importance (Random Forest)")
plt.show()